In [0]:
dbutils.widgets.text("raw_volume_path", "/Volumes/workspace/default/books_raw/", "Raw data volume path")
dbutils.widgets.text("target_schema", "workspace.default", "Target catalog.schema")

raw_volume_path = dbutils.widgets.get("raw_volume_path")
target_schema = dbutils.widgets.get("target_schema")

In [0]:
import logging, sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO,
                     format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("book_etl")

def log_step(step_name, row_count=None, extra=None):
    msg = f"STEP: {step_name}"
    if row_count is not None:
        msg += f" | rows={row_count}"
    if extra:
        msg += f" | {extra}"
    logger.info(msg)

In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiline", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(f"{raw_volume_path}books_enriched.csv")

In [0]:
def validate_schema(df, expected_columns, table_name):
    actual = set(df.columns)
    expected = set(expected_columns)
    missing = expected - actual
    if missing:
        raise ValueError(f"[{table_name}] Missing expected columns: {missing}")
    log_step(f"schema_validated_{table_name}", df.count())

In [0]:
validate_schema(df, [
    "book_id", "title", "authors", "average_rating", "ratings_count",
    "genres", "description", "original_publication_year", "pages"
], "books")

In [0]:
display(df.limit(10))
df.count()

In [0]:
total = df.count()
unique = df.dropDuplicates().count()
log_step("books_duplicate_check", total, f"unique={unique}, duplicates={total - unique}")

In [0]:
missing_desc = df.filter(df.description.isNull()).count()
print(f"Missing descriptions: {missing_desc}")

In [0]:
df.describe()

In [0]:
display(df.select("_c0", "index", "book_id").limit(10))

In [0]:
df.filter(df._c0 != df.index).count()

In [0]:
display(df.filter(df._c0 != df.index).select("_c0", "index", "book_id", "title").limit(10))

In [0]:
df = df.drop("_c0", "index")

In [0]:
display(df.limit(10))

In [0]:
def quality_gate(df, table_name, key_column, min_expected_rows=1):
    row_count = df.count()
    null_keys = df.filter(df[key_column].isNull()).count()
    duplicate_keys = row_count - df.dropDuplicates([key_column]).count()
    log_step(f"quality_check_{table_name}", row_count,
              f"null_keys={null_keys}, duplicate_keys={duplicate_keys}")
    if row_count < min_expected_rows:
        raise ValueError(f"[{table_name}] Row count {row_count} below minimum {min_expected_rows} — aborting.")
    if null_keys > 0:
        raise ValueError(f"[{table_name}] Found {null_keys} null values in key column '{key_column}' — aborting.")
    if duplicate_keys > 0:
        logger.warning(f"[{table_name}] Found {duplicate_keys} duplicate book_ids — deduplicating.")
        df = df.dropDuplicates([key_column])
    return df

df = quality_gate(df, "books", key_column="book_id", min_expected_rows=9000)

In [0]:
from delta.tables import DeltaTable

def upsert_delta(spark, source_df, target_table_name, merge_key):
    if not spark.catalog.tableExists(target_table_name):
        source_df.write.format("delta").saveAsTable(target_table_name)
        log_step(f"created_table_{target_table_name}", source_df.count())
        return

    target = DeltaTable.forName(spark, target_table_name)
    (
        target.alias("t")                     # "t" = the existing table (target)
        .merge(
            source_df.alias("s"),              # "s" = the new incoming data (source)
            f"t.{merge_key} = s.{merge_key}"    # the matching condition: same book_id = same book
        )
        .whenMatchedUpdateAll()                 # if book_id already exists -> update all its columns
        .whenNotMatchedInsertAll()               # if book_id is new -> insert it as a new row
        .execute()
    )
    log_step(f"merged_{target_table_name}", source_df.count())

In [0]:
print(repr(target_schema))

In [0]:
#df.write.format("delta").mode("overwrite").saveAsTable(f"{target_schema}.books")
#log_step("write_books_complete", df.count())

upsert_delta(spark, df, f"{target_schema}.books", merge_key="book_id")

In [0]:
def upsert_delta_composite(spark, source_df, target_table_name, merge_keys):
    # Same idea as upsert_delta, but the "is this the same row" check
    # needs to compare MULTIPLE columns at once, not just one.

    if not spark.catalog.tableExists(target_table_name):
        source_df.write.format("delta").saveAsTable(target_table_name)
        log_step(f"created_table_{target_table_name}", source_df.count())
        return

    target = DeltaTable.forName(spark, target_table_name)

    # Build a condition like: "t.user_id = s.user_id AND t.book_id = s.book_id"
    merge_condition = " AND ".join([f"t.{k} = s.{k}" for k in merge_keys])
    (
        target.alias("t")
        .merge(source_df.alias("s"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    log_step(f"merged_{target_table_name}", source_df.count())

In [0]:
ratings_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{raw_volume_path}ratings.csv")
display(ratings_df.limit(10))
ratings_df.count()

In [0]:
validate_schema(ratings_df, ["user_id", "book_id", "rating"], "ratings")
log_step("read_ratings", ratings_df.count())

In [0]:
total = ratings_df.count()
unique_pairs = ratings_df.select("user_id", "book_id").distinct().count()
print(f"Total ratings: {total}, unique user/book pairs: {unique_pairs}, duplicates: {total - unique_pairs}")

In [0]:
ratings_df.select("rating").distinct().orderBy("rating").show()

In [0]:
orphans = ratings_df.join(df, "book_id", "left_anti").count()
print(f"Ratings with no matching book: {orphans}")

In [0]:
def quality_gate_composite(df, table_name, key_columns, min_expected_rows=1):
    row_count = df.count()
    null_keys = df.filter(" OR ".join([f"{c} IS NULL" for c in key_columns])).count()
    duplicate_keys = row_count - df.dropDuplicates(key_columns).count()
    log_step(f"quality_check_{table_name}", row_count,
              f"null_keys={null_keys}, duplicate_keys={duplicate_keys}")
    if row_count < min_expected_rows:
        raise ValueError(f"[{table_name}] Row count {row_count} below minimum {min_expected_rows} — aborting.")
    if null_keys > 0:
        raise ValueError(f"[{table_name}] Found {null_keys} null values in key columns {key_columns} — aborting.")
    if duplicate_keys > 0:
        logger.warning(f"[{table_name}] Found {duplicate_keys} duplicate (user_id, book_id) pairs — deduplicating.")
        df = df.dropDuplicates(key_columns)
    return df

ratings_df = quality_gate_composite(ratings_df, "ratings", key_columns=["user_id", "book_id"], min_expected_rows=900000)

In [0]:
#ratings_df.write.format("delta").mode("overwrite").saveAsTable(f"{target_schema}.ratings")
#log_step("write_ratings_complete", ratings_df.count())

upsert_delta_composite(spark, ratings_df, f"{target_schema}.ratings", merge_keys=["user_id", "book_id"])

In [0]:
#book_tags is a small reference table fully rebuilt each run

In [0]:
book_tags_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{raw_volume_path}book_tags.csv")
display(book_tags_df.limit(10))
book_tags_df.count()

In [0]:
validate_schema(book_tags_df, ["goodreads_book_id", "tag_id", "count"], "book_tags")
log_step("read_book_tags", book_tags_df.count())

In [0]:
before = book_tags_df.count()
book_tags_df = book_tags_df.dropDuplicates(["goodreads_book_id", "tag_id"])
after = book_tags_df.count()
log_step("book_tags_deduplicated", after, f"removed={before - after}")

In [0]:
orphans = book_tags_df.join(df, "goodreads_book_id", "left_anti").count()
print(f"Tag rows with no matching book: {orphans}")

In [0]:
total = book_tags_df.count()
unique_pairs = book_tags_df.select("goodreads_book_id", "tag_id").distinct().count()
print(f"Total: {total}, Unique book/tag pairs: {unique_pairs}, Duplicates: {total - unique_pairs}")

In [0]:
from pyspark.sql import functions as F
dupe_pairs = book_tags_df.groupBy("goodreads_book_id", "tag_id").count().filter(F.col("count") > 1)
display(dupe_pairs)

In [0]:
book_tags_df.filter((book_tags_df.goodreads_book_id == 52629) & (book_tags_df.tag_id == 10094)).show()

In [0]:
book_tags_df = book_tags_df.dropDuplicates(["goodreads_book_id", "tag_id"])
book_tags_df.count()

In [0]:
book_tags_df.write.format("delta").mode("overwrite").saveAsTable(f"{target_schema}.book_tags")
log_step("write_book_tags_complete", book_tags_df.count())

In [0]:
log_step("etl_run_complete", extra="All tables (books, ratings, book_tags) processed successfully.")

In [0]:
print("Books after run 1:", spark.table(f"{target_schema}.books").count())
print("Ratings after run 1:", spark.table(f"{target_schema}.ratings").count())

In [0]:
print("Books after run 2:", spark.table(f"{target_schema}.books").count())
print("Ratings after run 2:", spark.table(f"{target_schema}.ratings").count())